## Step 1: Base Model Assessment (Beyond PR-AUC)

In [13]:
import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
#load models
model_B_path = 'Models B/lightgbm_model.txt'
model_A_path = 'Model A/cardio_logreg_pipeline_cvd.joblib'

# Wrap LightGBM Booster to match sklearn interface
class LGBWrapper:
    def __init__(self, booster):
        self.booster = booster
    
    def predict_proba(self, X):
        proba_pos = self.booster.predict(X)
        return np.column_stack([1 - proba_pos, proba_pos])


model_B = LGBWrapper(lgb.Booster(model_file=model_B_path))
#model_B = joblib.load('Models B/random_forest.joblib')
model_A = joblib.load(model_A_path)


d:\ACADEMY FOLDER\IUT Academics\Semester 6 (3.2)\ML\Hybrid-CVD-Model\.venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.4.1.post1 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\ACADEMY FOLDER\IUT Academics\Semester 6 (3.2)\ML\Hybrid-CVD-Model\.venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.4.1.post1 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\ACADEMY FOLDER\IUT Academics\Semester 6 (3.2)\ML\Hybrid-CVD-Model\.venv\Lib\site-packages\skle

In [16]:
#Age,Alcohol Intake,BMI,Cardiovascular Disease,Cholesterol Level,Diastolic Blood Pressure,Gender,Glucose Level,Physical Activity,Smoking Status,Systolic Blood Pressure,id


from sklearn.model_selection import train_test_split
import pandas as pd

dataset_path = 'Meta Model Dataset/Training_For_Meta_Model.csv'
dataset = pd.read_csv(dataset_path)

# Split dataset into features and target
X = dataset.drop(['Cardiovascular Disease','id'], axis=1)
y = dataset['Cardiovascular Disease']

X_train_health, X_test_health, y_train_health, y_test_health = train_test_split(X[['Age','BMI','Cholesterol Level','Diastolic Blood Pressure','Gender','Glucose Level','Systolic Blood Pressure']], y, test_size=0.2, random_state=42)
X_train_lifestyle, X_test_lifestyle, y_train_lifestyle, y_test_lifestyle = train_test_split(X[['Alcohol Intake','Cholesterol Level','Physical Activity','Smoking Status','Gender','Age','BMI']], y, test_size=0.2, random_state=42)

In [17]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
import numpy as np

def evaluate_base_model_for_ensemble(model, X_test, y_test, model_name):
    """Evaluate base model for ensemble suitability"""
    y_proba = model.predict_proba(X_test)[:, 1]
    
    # 1. Probability calibration
    fraction_of_positives, mean_predicted_value = calibration_curve(
        y_test, y_proba, n_bins=10
    )
    
    # 2. Brier score (lower is better)
    brier = brier_score_loss(y_test, y_proba)
    
    # 3. Information content (entropy)
    entropy = -np.mean(y_proba * np.log(y_proba + 1e-15) + 
                      (1-y_proba) * np.log(1-y_proba + 1e-15))
    
    print(f"{model_name}:")
    print(f"  Brier Score: {brier:.4f}")
    print(f"  Entropy (Info Content): {entropy:.4f}")
    print(f"  Mean Probability: {np.mean(y_proba):.4f}")
    
    return {
        'probabilities': y_proba,
        'brier_score': brier,
        'entropy': entropy,
        'calibration': (fraction_of_positives, mean_predicted_value)
    }

# Evaluate your models
model_a_eval = evaluate_base_model_for_ensemble(model_A, X_test_lifestyle, y_test_lifestyle, "Model A (Lifestyle)")
model_b_eval = evaluate_base_model_for_ensemble(model_B, X_test_health, y_test_health, "Model B (Health)")

Model A (Lifestyle):
  Brier Score: 0.2413
  Entropy (Info Content): 0.6278
  Mean Probability: 0.4516
Model B (Health):
  Brier Score: 0.1803
  Entropy (Info Content): 0.5406
  Mean Probability: 0.5142


## Step 2: Information Complementarity Analysis

In [18]:
# Check if models provide complementary information
correlation = np.corrcoef(model_a_eval['probabilities'], 
                         model_b_eval['probabilities'])[0,1]

print(f"Probability Correlation: {correlation:.3f}")

# Ideal range: 0.3-0.7 (some correlation but still complementary)
if correlation < 0.3:
    print("✓ Models provide highly complementary information")
elif correlation < 0.7:
    print("✓ Good balance of correlation and complementarity") 
else:
    print("⚠ Models may be too similar - consider feature engineering")


Probability Correlation: 0.420
✓ Good balance of correlation and complementarity


## Step 3: Meta-Learner Training and Evaluation

In [ ]:
# Prepare meta-learner training data
meta_X = np.column_stack([model_a_eval['probabilities'], 
                          model_b_eval['probabilities']])

# Train meta-learner (your neural network)
meta_model = tf.keras.Sequential([
    tf.keras.layers.Dense(8, activation='relu', input_shape=(2,)),
    tf.keras.layers.Dense(4, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

meta_model.compile(optimizer='adam', loss='binary_crossentropy')
meta_model.fit(meta_X, y_test, epochs=50, validation_split=0.2)

# Final ensemble predictions
final_probabilities = meta_model.predict(meta_X).flatten()

# Evaluate the FINAL ensemble
from sklearn.metrics import precision_recall_curve, auc
precision, recall, _ = precision_recall_curve(y_test, final_probabilities)
final_pr_auc = auc(recall, precision)

print(f"Final Ensemble PR-AUC: {final_pr_auc:.4f}")
